In [1]:
import os
import pandas as pd
import numpy as np
import pandas_ta_classic
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
import tensorflow as tf
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBRegressor
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import warnings

os.makedirs('results', exist_ok=True)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
tf.config.experimental.enable_op_determinism()
import matplotlib.pyplot as plt
ARCHS = ['BiRNN', 'GRU', 'LSTM']
UNITS = [16, 32, 64, 128, 256]
SEEDS = [1, 2, 3]
COMBOS = {'B,G,L': [0, 1, 2], 'B,G': [0, 1], 'B,L': [0, 2], 'G,L': [1, 2]}
META = ['ANN', 'GRU', 'LSTM']
CHOSEN = {'B': ('BiRNN', 128), 'G': ('GRU', 16), 'L': ('LSTM', 16)}

SYMBOLS = {'ADC': 2140, 'ALRAJHI': 1120, 'EMAAR EC': 4220, 'FITAIHI': 4180, 'GACO': 6020,
           'SARCO': 2030, 'SEC': 5110, 'SPIMACO': 2070, 'STC': 7010, 'TASNEE': 2060}
EPOCHS = 1000
FEATURES = ['open', 'high', 'low', 'close',
    'alma_5_6.0_0.85', 'ldecay_5', 'dema_5', 'ema_5', 'fwma_5', 'hma_5', 'hl2', 'hlc3',
    'hwma_0.2_0.1_0.1', 'jma_5_0', 'kama_5_2_30', 'lr_5', 'median_5', 'midpoint_5',
    'midprice_5', 'pwma_5', 'qtl_5_0.5', 'rma_5', 'sinwma_5', 'sma_5', 'ssf_5_2',
    'swma_5', 't3_5_0.7', 'tema_5', 'trima_5', 'vwap_d', 'vwma_5', 'wcp', 'wma_5', 'zl_ema_5']


I0000 00:00:1793938705.525129 1470447 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793938705.635955 1470447 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793938706.410469 1470447 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Decisions vs. paper
- Step 1: zero-volume rows dropped; missing O/H/L filled from prev/today's close (paper: ffill)
- Step 3: library naming; 105 candidates pass vs. paper's 34 → paper's 34 used
- Step 7: shuffle=True; epochs ... (decided on val, see below)

In [2]:
raw = pd.read_csv('data/Tadawul_stcks.csv')
raw.columns = raw.columns.str.strip()
raw['date'] = pd.to_datetime(raw['date'])


In [30]:
flat = lambda Xw: Xw.reshape(len(Xw), -1)
def rw_paper_pred(Xw, h=5):
    """Authors' random walk: drift = mean change over the window's last 5 closes."""
    c = Xw[:, :, FEATURES.index('close')]
    drift = np.diff(c, axis=1).mean(axis=1, keepdims=True)
    return c[:, -1:] + drift * np.arange(1, h + 1)

def arima_paper_pred(y_fit, n_windows, order=(5, 1, 0), h=5):
    """Authors' ARIMA: one forecast from the end of train, repeated for every window."""
    f = ARIMA(y_fit.values, order=order).fit().forecast(h)
    return np.tile(f, (n_windows, 1))
def load_ticker(raw,symbol,calendar_ffill=False):
    df = raw[raw['symbol'] == symbol].copy()
    df = df.set_index('date').sort_index()
    assert df.index.is_unique
    df = df.rename(columns={'volume_traded': 'volume'})

    if calendar_ffill:   # authors' code: every calendar day, all columns forward-filled
        days = pd.date_range(df.index.min(), df.index.max(), freq='D', name='date')
        df = df.reindex(days).ffill()[['open', 'high', 'low', 'close', 'volume']]
    else:                # trading days only (current behaviour)
        df = df[df['volume'] > 0]
        prev = df['close'].shift(1).fillna(df['close'])
        df['open'] = df['open'].fillna(prev)
        df['high'] = df['high'].fillna(np.maximum(prev, df['close']))
        df['low']  = df['low'].fillna(np.minimum(prev, df['close']))
        df = df[['open', 'high', 'low', 'close', 'volume']]

    assert not df.isna().any().any()
    return df

def split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Chronological train / val / test on the configured dates (bounds inclusive). Never shuffled."""
    train = df.loc[:'2015-12-31']
    val = df.loc['2016-01-01':'2016-12-31']
    test = df.loc['2017-01-01':'2020-03-05']
    return train, val, test

def all_indicators(df):
    parts = [df[['open','high','low','close','volume']]]
    failed = {}
    for name in df.ta.indicators(as_list=True):
        try:
            out = getattr(df.ta,name)(length=5, lookahead = False)
        except Exception as e:
            failed[name] = type(e).__name__
            continue
        if out is None:
            continue
        elif isinstance(out, pd.Series):
            # out.name = out.name.split("_", 1)[0]
            out.name = out.name.lower()
        elif isinstance(out, pd.DataFrame):
            # out.columns =  out.columns.str.split('_').str[0]
            out.columns =  out.columns.str.lower()
        parts.append(out)
    X = pd.concat(parts,axis=1)
    X = X.loc[:, ~X.columns.duplicated()]
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.dropna(axis=1, how='all')
    assert X.columns.is_unique
    return X, failed

def prepare(raw,symbol,calendar_ffill=False):
    X = all_indicators(load_ticker(raw,symbol,calendar_ffill))[0][FEATURES].dropna()
    train, val, test = split(X)
    x_scaler = StandardScaler().fit(train[FEATURES])
    y_scaler = StandardScaler().fit(train[["close"]])
    def scale(part):
        xs = pd.DataFrame(x_scaler.transform(part[FEATURES]), index=part.index, columns=FEATURES)
        ys = pd.Series(y_scaler.transform(part[['close']]).ravel(), index=part.index, name='y')
        return xs, ys
    return scale(train), scale(val), scale(test), x_scaler, y_scaler
def make_windows(X,y,w=5,h=5):
    n = len(X) - w - h + 1
    Xw = np.stack([X[i:i + w] for i in range(n)])
    Yw = np.stack([y[i + w:i + w + h] for i in range(n)])
    return Xw, Yw

def build_model(arch, units, lr=1e-3, w=5, n_features=34, h=5):
    layer = {
        'GRU':   lambda: keras.layers.GRU(units, activation='relu'),
        'LSTM':  lambda: keras.layers.LSTM(units, activation='relu'),
        'BiRNN': lambda: keras.layers.Bidirectional(keras.layers.SimpleRNN(units, activation='relu')),
    }[arch]()
    model = keras.Sequential([keras.Input(shape=(w, n_features)), layer, keras.layers.Dense(h)])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='mse')
    return model
def fit(model, Xtr, ytr, Xva, yva, epochs, label=''):
    stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20,
                                         restore_best_weights=True)
    history = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                        epochs=epochs, batch_size=int(0.1 * len(Xtr)), shuffle=True,
                        callbacks=[stop], verbose=0)
    print(f'{label:16s} epochs {len(history.history["loss"]):4d}  val {min(history.history["val_loss"]):.6f}')
    return history, model

def train(arch, units, epochs, Xtr, ytr, Xva, yva, seed=42):
    keras.utils.set_random_seed(seed)
    model = build_model(arch, units, 1e-3)
    return fit(model, Xtr, ytr, Xva, yva, epochs, label=f'{arch} {units}')
def metrics(pred, y, y_scaler):
    p = y_scaler.inverse_transform(pred.reshape(-1, 1)).reshape(pred.shape)
    t = y_scaler.inverse_transform(y.reshape(-1, 1)).reshape(y.shape)
    e = p - t
    return {
        'MAE':  np.mean(np.abs(e)),
        'MSE':  np.mean(e ** 2),
        'RMSE': np.sqrt(np.mean(e ** 2)),
        'MRE':  np.mean(np.abs(e) / np.abs(t)),
    }
def build_meta(kind, n_base, lr=1e-3, h=5):
    body = {
        'ANN':  lambda: [keras.layers.Flatten(),
                         keras.layers.Dense(128, activation='relu'),
                         keras.layers.Dense(64, activation='relu'),
                         keras.layers.Dense(32, activation='relu')],
        'GRU':  lambda: [keras.layers.GRU(128, activation='relu')],
        'LSTM': lambda: [keras.layers.LSTM(128, activation='relu')],
    }[kind]()
    model = keras.Sequential([keras.Input(shape=(h, n_base)), *body, keras.layers.Dense(h)])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='mse')
    return model
def oof_folds(n, n_blocks=5, first=0.4, n_es=250, gap=5):
    """
    Walk-forward folds over n training windows.

        |------ fit ------| gap |--- es ---| gap |-- block --|
          train the model        early stop       predict (OOF)

    The first 40% is never predicted; the rest is cut into n_blocks blocks.
    """
    cuts = np.linspace(int(n * first), n, n_blocks + 1).astype(int)

    for block_start, block_end in zip(cuts[:-1], cuts[1:]):
        es_end   = block_start - gap        # leave a gap before the block
        es_start = es_end - n_es            # ~1 year for early stopping
        fit_end  = es_start - gap           # leave a gap before early stopping

        fit   = slice(0, fit_end)
        es    = slice(es_start, es_end)
        block = slice(block_start, block_end)
        yield fit, es, block
def naive_pred(Xw, h=5):
    ci = FEATURES.index('close')
    return np.repeat(Xw[:, -1, ci:ci+1], h, axis=1)

def naive_mse(Xw, yw):
    return np.mean((naive_pred(Xw) - yw) ** 2)        
def run_stacking(X_meta, y_meta, X_meta_va, y_va, tag=''):
    out, models = {}, {}
    for combo, cols in COMBOS.items():
        for kind in META:
            name = f'STK_{kind}({combo}){tag}'
            keras.utils.set_random_seed(42)
            meta = build_meta(kind, len(cols))
            _, meta = fit(meta, X_meta[:, :, cols], y_meta,
                          X_meta_va[:, :, cols], y_va, EPOCHS, label=name)
            out[name] = meta.predict(X_meta_va[:, :, cols], verbose=0)
            models[name] = (meta, cols)
    return out, models
def drift_pred(Xw, ytr, h=5):
    d = np.diff(ytr).mean()                       # mean daily change, train only
    return naive_pred(Xw, h) + d * np.arange(1, h + 1)

def arima_pred(y_fit, y_target, order, y_between=None, w=5, h=5):
    res = ARIMA(y_fit.values, order=order).fit()
    before = [y_fit] if y_between is None else [y_fit, y_between]
    full = res.apply(np.concatenate([s.values for s in before + [y_target]]))
    offset = sum(len(s) for s in before)
    out = []
    for i in range(len(y_target) - w - h + 1):
        origin = offset + i + w - 1
        out.append(full.get_prediction(start=origin + 1, end=origin + h, dynamic=True).predicted_mean)
    return np.array(out)
def window_rmse(pred, y, y_scaler, step=5):
    p = y_scaler.inverse_transform(pred.reshape(-1, 1)).reshape(pred.shape)
    t = y_scaler.inverse_transform(y.reshape(-1, 1)).reshape(y.shape)
    return np.sqrt(((p - t) ** 2).mean(axis=1))[::step]

def win_loss(preds, y, y_scaler, alpha=0.05):
    r = {k: window_rmse(p, y, y_scaler) for k, p in preds.items()}
    pairs = list(combinations(r, 2))
    pvals = [wilcoxon(r[a], r[b]).pvalue for a, b in pairs]
    reject = multipletests(pvals, alpha=alpha, method='holm')[0]
    score = {k: {'win': 0, 'loss': 0, 'tie': 0} for k in r}
    for (a, b), sig in zip(pairs, reject):
        if not sig:
            score[a]['tie'] += 1; score[b]['tie'] += 1
        elif np.median(r[a] - r[b]) < 0:
            score[a]['win'] += 1; score[b]['loss'] += 1
        else:
            score[b]['win'] += 1; score[a]['loss'] += 1
    df = pd.DataFrame(score).T
    df['win_rate'] = df['win'] / (df['win'] + df['loss']).replace(0, np.nan)
    return df.sort_values(['win', 'loss'], ascending=[False, True])

def add_averages(p):
    p['AVG(B,G,L)'] = (p['B'] + p['G'] + p['L']) / 3
    p['AVG(B,G)']   = (p['B'] + p['G']) / 2
    p['AVG(B,L)']   = (p['B'] + p['L']) / 2
    p['AVG(G,L)']   = (p['G'] + p['L']) / 2
    return p

def oof_predictions(Xwtr, ywtr, seed=42):
    X_blocks, y_blocks = [], []
    for fit_sl, es_sl, block_sl in oof_folds(len(Xwtr)):
        block_preds = []
        for arch, units in CHOSEN.values():
            _, m = train(arch, units, EPOCHS, Xwtr[fit_sl], ywtr[fit_sl],
                         Xwtr[es_sl], ywtr[es_sl], seed=seed)
            block_preds.append(m.predict(Xwtr[block_sl], verbose=0))
        X_blocks.append(np.stack(block_preds, axis=-1))
        y_blocks.append(ywtr[block_sl])
    return np.concatenate(X_blocks), np.concatenate(y_blocks)

def run_ticker(symbol, seed=42, calendar_ffill=False ):
    keras.backend.clear_session()                        # free memory from the previous ticker
    (xtr, ytr), (xva, yva), (xte, yte), _, ys = prepare(raw, symbol,calendar_ffill)
    Xwtr, ywtr = make_windows(xtr, ytr)
    Xwva, ywva = make_windows(xva, yva)
    Xwte, ywte = make_windows(xte, yte)

    # base models + averages
    base = {k: train(a, u, EPOCHS, Xwtr, ywtr, Xwva, ywva, seed)[1] for k, (a, u) in CHOSEN.items()}
    pv = add_averages({k: m.predict(Xwva, verbose=0) for k, m in base.items()})
    pt = add_averages({k: m.predict(Xwte, verbose=0) for k, m in base.items()})

    # stacking on full OOF
    X_meta, y_meta = oof_predictions(Xwtr, ywtr, seed)
    meta_va = np.stack([pv['B'], pv['G'], pv['L']], axis=-1)
    meta_te = np.stack([pt['B'], pt['G'], pt['L']], axis=-1)
    stk_va, stk_models = run_stacking(X_meta, y_meta, meta_va, ywva)
    pv |= stk_va
    for name, (meta, cols) in stk_models.items():
        pt[name] = meta.predict(meta_te[:, :, cols], verbose=0)

    # baselines
    pv['Naive'], pt['Naive'] = naive_pred(Xwva), naive_pred(Xwte)
    pv['RW drift'], pt['RW drift'] = drift_pred(Xwva, ytr), drift_pred(Xwte, ytr)
    
    pv['RW (paper)'], pt['RW (paper)'] = rw_paper_pred(Xwva), rw_paper_pred(Xwte)
    pv['ARIMA (paper)'] = arima_paper_pred(ytr, len(Xwva))
    pt['ARIMA (paper)'] = arima_paper_pred(ytr, len(Xwte))
    aic = {(p, 1, q): ARIMA(ytr.values, order=(p, 1, q)).fit().aic for p in range(3) for q in range(3)}
    order = min(aic, key=aic.get)
    pv['ARIMA'] = arima_pred(ytr, yva, order)
    pt['ARIMA'] = arima_pred(ytr, yte, order, y_between=yva)

    xgb = XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=4,
                       tree_method='hist', early_stopping_rounds=50, random_state=seed)
    xgb.fit(flat(Xwtr), ywtr, eval_set=[(flat(Xwva), ywva)], verbose=False)
    pv['XGBoost'], pt['XGBoost'] = xgb.predict(flat(Xwva)), xgb.predict(flat(Xwte))

    _, gru256 = train('GRU', 256, EPOCHS, Xwtr, ywtr, Xwva, ywva, seed)
    pv['GRU-256'], pt['GRU-256'] = gru256.predict(Xwva, verbose=0), gru256.predict(Xwte, verbose=0)

    val  = pd.DataFrame({k: metrics(p, ywva, ys) for k, p in pv.items()}).T
    test = pd.DataFrame({k: metrics(p, ywte, ys) for k, p in pt.items()}).T
    return val, test, win_loss(pt, ywte, ys), order

In [4]:
alrajhi = load_ticker(raw,1120)
(xtr, ytr), (xva, yva), (xte, yte), xs, ys = prepare(raw, 1120, calendar_ffill=True)
Xwtr, ywtr = make_windows(xtr, ytr)
Xwva, ywva = make_windows(xva, yva)

In [5]:
print(xtr.mean().abs().max())
print(xtr.std().mean())
print(xva.mean().mean())
print(xte.mean().mean())

3.8947234435573273e-16
1.00009824630372
-0.2887296337802478
0.5524829489084958


In [6]:
# np.allclose(ys.inverse_transform(ytr.to_frame()).ravel(), alrajhi.loc[ytr.index, 'close'])

In [7]:
print(Xwtr.shape)
print(ywtr.shape)

(5081, 5, 34)
(5081, 5)


In [8]:
print('naive', naive_mse(Xwva, ywva))
              

naive 0.001191564749853788


In [9]:
base = {}
for k, (arch, units) in CHOSEN.items():
    _, base[k] = train(arch, units, EPOCHS, Xwtr, ywtr, Xwva, ywva, seed=42)

W0000 00:00:1793938710.114165 1470447 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1793938710.118419 1470447 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1793938710.260206 1470447 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13209 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a
E0000 00:00:1793938711.129816 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=out

BiRNN 128        epochs  118  val 0.001302


E0000 00:00:1793938745.897195 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


GRU 16           epochs  319  val 0.001216


E0000 00:00:1793938845.125851 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


LSTM 16          epochs  469  val 0.001233


In [10]:
preds = {k: m.predict(Xwva, verbose=0) for k, m in base.items()}
preds['Naive'] = naive_pred(Xwva)

E0000 00:00:1793938946.774966 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


In [11]:
# preds['AVG(B,G,L)'] = (preds['B'] + preds['G'] + preds['L']) / 3
# preds['AVG(B,G)']   = (preds['B'] + preds['G']) / 2
# preds['AVG(B,L)']   = (preds['B'] + preds['L']) / 2
# preds['AVG(G,L)']   = (preds['G'] + preds['L']) / 2
preds = add_averages(preds)

In [12]:
table = pd.DataFrame({k: metrics(p, ywva, ys) for k, p in preds.items()}).T
print(table.round(4).sort_values('RMSE'))

               MAE     MSE    RMSE     MRE
Naive       0.4897  0.5290  0.7273  0.0137
AVG(G,L)    0.4934  0.5332  0.7302  0.0139
AVG(B,G,L)  0.4967  0.5337  0.7306  0.0139
AVG(B,G)    0.5029  0.5381  0.7335  0.0141
G           0.5037  0.5400  0.7348  0.0142
AVG(B,L)    0.5095  0.5460  0.7389  0.0143
L           0.5052  0.5476  0.7400  0.0142
B           0.5381  0.5782  0.7604  0.0150


In [13]:
X_blocks, y_blocks = [], []

for fit_sl, es_sl, block_sl in oof_folds(len(Xwtr)):
    print('fit', fit_sl, ' es', es_sl, ' block', block_sl)

    block_preds = []
    for arch, units in CHOSEN.values():
        _, model = train(arch, units, EPOCHS,
                         Xwtr[fit_sl], ywtr[fit_sl],
                         Xwtr[es_sl],  ywtr[es_sl],
                         seed=42)
        block_preds.append(model.predict(Xwtr[block_sl], verbose=0))

    X_blocks.append(np.stack(block_preds, axis=-1))
    y_blocks.append(ywtr[block_sl])

X_meta = np.concatenate(X_blocks)
y_meta = np.concatenate(y_blocks)

fit slice(0, np.int64(1772), None)  es slice(np.int64(1777), np.int64(2027), None)  block slice(np.int64(2032), np.int64(2641), None)


E0000 00:00:1793939358.910810 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


BiRNN 128        epochs   52  val 0.007207


E0000 00:00:1793939374.769591 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


GRU 16           epochs  126  val 0.008345


E0000 00:00:1793939414.427593 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


... (22 outputs left out)

GRU 16           epochs  367  val 0.000804


E0000 00:00:1793939971.687151 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


LSTM 16          epochs  398  val 0.000788


E0000 00:00:1793940060.168480 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


In [14]:
for f, (Xb, yb, (_, _, block)) in enumerate(zip(X_blocks, y_blocks, oof_folds(len(Xwtr)))):
    row = {k: metrics(Xb[:, :, j], yb, ys)['RMSE'] for j, k in enumerate(CHOSEN)}
    row['Naive'] = metrics(naive_pred(Xwtr[block]), yb, ys)['RMSE']
    print(f, {k: round(float(v), 3) for k, v in row.items()})

0 {'B': 2.574, 'G': 2.559, 'L': 2.778, 'Naive': 2.219}
1 {'B': 1.044, 'G': 1.163, 'L': 1.01, 'Naive': 0.868}
2 {'B': 0.838, 'G': 0.914, 'L': 0.844, 'Naive': 0.81}
3 {'B': 0.672, 'G': 0.718, 'L': 0.634, 'Naive': 0.603}
4 {'B': 1.138, 'G': 1.006, 'L': 0.998, 'Naive': 0.969}


In [15]:
table = pd.DataFrame({k: metrics(p, ywva, ys) for k, p in preds.items()}).T
print(table.round(4).sort_values('RMSE'))

               MAE     MSE    RMSE     MRE
Naive       0.4897  0.5290  0.7273  0.0137
AVG(G,L)    0.4934  0.5332  0.7302  0.0139
AVG(B,G,L)  0.4967  0.5337  0.7306  0.0139
AVG(B,G)    0.5029  0.5381  0.7335  0.0141
G           0.5037  0.5400  0.7348  0.0142
AVG(B,L)    0.5095  0.5460  0.7389  0.0143
L           0.5052  0.5476  0.7400  0.0142
B           0.5381  0.5782  0.7604  0.0150


In [16]:
X_meta_va = np.stack([preds['B'], preds['G'], preds['L']], axis=-1)
stk_va, stk_models = run_stacking(X_meta, y_meta, X_meta_va, ywva)
preds |= stk_va

STK_ANN(B,G,L)   epochs   34  val 0.001197


E0000 00:00:1793940065.247695 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


STK_GRU(B,G,L)   epochs   43  val 0.001212


E0000 00:00:1793940079.683491 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


STK_LSTM(B,G,L)  epochs   48  val 0.001202


E0000 00:00:1793940091.212782 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


... (14 outputs left out)

STK_GRU(G,L)     epochs   31  val 0.001214


E0000 00:00:1793940161.803612 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


STK_LSTM(G,L)    epochs   28  val 0.001218


E0000 00:00:1793940169.119568 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


In [17]:
start = len(X_blocks[0])
stk_va_trim, stk_models_trim = run_stacking(X_meta[start:], y_meta[start:], X_meta_va, ywva, tag=' trim')
preds |= stk_va_trim

STK_ANN(B,G,L) trim epochs   31  val 0.001204
STK_GRU(B,G,L) trim epochs   29  val 0.001202


E0000 00:00:1793940177.482671 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


STK_LSTM(B,G,L) trim epochs   81  val 0.001197


E0000 00:00:1793940188.462666 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


STK_ANN(B,G) trim epochs   35  val 0.001192
STK_GRU(B,G) trim epochs   31  val 0.001214


E0000 00:00:1793940197.308969 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


... (6 outputs left out)

STK_ANN(G,L) trim epochs   39  val 0.001191
STK_GRU(G,L) trim epochs   31  val 0.001210


E0000 00:00:1793940228.137072 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


STK_LSTM(G,L) trim epochs   43  val 0.001189


E0000 00:00:1793940234.393291 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


In [18]:
table = pd.DataFrame({k: metrics(p, ywva, ys) for k, p in preds.items()}).T
stk = table[table.index.str.startswith('STK')]['RMSE']
print(stk.sort_values().round(4))

STK_LSTM(B,G) trim      0.7241
STK_ANN(B,G)            0.7263
STK_LSTM(G,L) trim      0.7264
STK_ANN(G,L) trim       0.7272
STK_ANN(B,G) trim       0.7273
STK_ANN(B,G,L)          0.7289
STK_LSTM(B,G,L) trim    0.7290
STK_GRU(B,G,L) trim     0.7305
STK_LSTM(B,G,L)         0.7306
STK_GRU(B,G)            0.7309
STK_ANN(G,L)            0.7311
STK_ANN(B,G,L) trim     0.7311
STK_LSTM(B,G)           0.7325
STK_GRU(G,L) trim       0.7330
STK_GRU(B,G,L)          0.7335
STK_ANN(B,L)            0.7339
STK_GRU(G,L)            0.7340
STK_GRU(B,G) trim       0.7340
STK_LSTM(B,L) trim      0.7352
STK_LSTM(G,L)           0.7353
STK_ANN(B,L) trim       0.7365
STK_LSTM(B,L)           0.7403
STK_GRU(B,L)            0.7430
STK_GRU(B,L) trim       0.7466
Name: RMSE, dtype: float64


In [19]:
xgb = XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=4,
                   tree_method='hist', early_stopping_rounds=50, random_state=42)
xgb.fit(flat(Xwtr), ywtr, eval_set=[(flat(Xwva), ywva)], verbose=False)
preds['XGBoost'] = xgb.predict(flat(Xwva))

In [20]:
print('XGBoost best iteration:', xgb.best_iteration)

XGBoost best iteration: 130


In [21]:
preds['RW drift'] = drift_pred(Xwva, ytr)
print('drift per day (scaled):', np.diff(ytr).mean())

drift per day (scaled): 0.0001983697752300953


In [22]:
print(np.allclose(arima_pred(ytr, yva, (0, 1, 0)), naive_pred(Xwva)))

True


In [23]:
aic = {}
for p in range(3):
    for q in range(3):
        aic[(p, 1, q)] = ARIMA(ytr.values, order=(p, 1, q)).fit().aic
order = min(aic, key=aic.get)
print(order, round(aic[order], 1))
preds['ARIMA'] = arima_pred(ytr, yva, order)

(0, 1, 2) -16041.9


In [24]:
_, gru256 = train('GRU', 256, EPOCHS, Xwtr, ywtr, Xwva, ywva, seed=42)
preds['GRU-256'] = gru256.predict(Xwva, verbose=0)

E0000 00:00:1793940240.061072 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


GRU 256          epochs  115  val 0.001247


E0000 00:00:1793940275.704349 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


In [25]:
table = pd.DataFrame({k: metrics(p, ywva, ys) for k, p in preds.items()}).T
print(table[~table.index.str.endswith('trim')].round(4).sort_values('RMSE'))

                    MAE     MSE    RMSE     MRE
RW drift         0.4890  0.5273  0.7261  0.0137
STK_ANN(B,G)     0.4941  0.5275  0.7263  0.0139
Naive            0.4897  0.5290  0.7273  0.0137
ARIMA            0.4946  0.5293  0.7275  0.0139
STK_ANN(B,G,L)   0.4963  0.5313  0.7289  0.0139
AVG(G,L)         0.4934  0.5332  0.7302  0.0139
AVG(B,G,L)       0.4967  0.5337  0.7306  0.0139
STK_LSTM(B,G,L)  0.4968  0.5338  0.7306  0.0140
STK_GRU(B,G)     0.5047  0.5342  0.7309  0.0141
STK_ANN(G,L)     0.4945  0.5345  0.7311  0.0139
STK_LSTM(B,G)    0.5002  0.5365  0.7325  0.0141
AVG(B,G)         0.5029  0.5381  0.7335  0.0141
STK_GRU(B,G,L)   0.5047  0.5380  0.7335  0.0141
STK_ANN(B,L)     0.5028  0.5386  0.7339  0.0141
STK_GRU(G,L)     0.4993  0.5388  0.7340  0.0140
G                0.5037  0.5400  0.7348  0.0142
STK_LSTM(G,L)    0.5100  0.5407  0.7353  0.0143
AVG(B,L)         0.5095  0.5460  0.7389  0.0143
L                0.5052  0.5476  0.7400  0.0142
STK_LSTM(B,L)    0.5077  0.5480  0.7403 

TEST

In [26]:
Xwte, ywte = make_windows(xte, yte)

# alignment check first, like on val
print(np.allclose(arima_pred(ytr, yte, (0, 1, 0), y_between=yva), naive_pred(Xwte)))

pt = {k: m.predict(Xwte, verbose=0) for k, m in base.items()}
pt['Naive']    = naive_pred(Xwte)
pt['RW drift'] = drift_pred(Xwte, ytr)
pt['ARIMA']    = arima_pred(ytr, yte, order, y_between=yva)
pt['XGBoost']  = xgb.predict(flat(Xwte))
pt['GRU-256']  = gru256.predict(Xwte, verbose=0)

pt['AVG(B,G,L)'] = (pt['B'] + pt['G'] + pt['L']) / 3
pt['AVG(B,G)']   = (pt['B'] + pt['G']) / 2
pt['AVG(B,L)']   = (pt['B'] + pt['L']) / 2
pt['AVG(G,L)']   = (pt['G'] + pt['L']) / 2

X_meta_te = np.stack([pt['B'], pt['G'], pt['L']], axis=-1)
for name, (meta, cols) in stk_models.items():
    pt[name] = meta.predict(X_meta_te[:, :, cols], verbose=0)

test_table = pd.DataFrame({k: metrics(p, ywte, ys) for k, p in pt.items()}).T
print(test_table.round(4).sort_values('RMSE'))

True


E0000 00:00:1793940322.672491 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


                    MAE     MSE    RMSE     MRE
RW drift         0.6862  1.0807  1.0396  0.0123
Naive            0.6855  1.0820  1.0402  0.0123
ARIMA            0.6910  1.0827  1.0405  0.0124
G                0.7340  1.1507  1.0727  0.0133
AVG(B,G,L)       0.7338  1.1530  1.0738  0.0132
AVG(G,L)         0.7419  1.1933  1.0924  0.0133
STK_GRU(G,L)     0.7812  1.2101  1.1000  0.0140
STK_ANN(B,G,L)   0.7744  1.2153  1.1024  0.0139
AVG(B,L)         0.7660  1.2196  1.1044  0.0137
AVG(B,G)         0.7769  1.2364  1.1119  0.0139
STK_GRU(B,L)     0.7982  1.2413  1.1141  0.0143
STK_LSTM(B,G,L)  0.7860  1.2651  1.1248  0.0140
STK_LSTM(B,L)    0.7827  1.2679  1.1260  0.0139
GRU-256          0.8234  1.3343  1.1551  0.0146
STK_ANN(G,L)     0.8330  1.3418  1.1584  0.0148
STK_GRU(B,G,L)   0.8269  1.3471  1.1607  0.0146
L                0.7958  1.3545  1.1638  0.0141
STK_GRU(B,G)     0.8409  1.3991  1.1828  0.0149
STK_LSTM(B,G)    0.8249  1.4232  1.1930  0.0146
XGBoost          0.8674  1.5487  1.2445 

In [39]:


print(win_loss(pt, ywte, ys))

                 win  loss  tie  win_rate
Naive             21     0    2  1.000000
RW drift          21     0    2  1.000000
ARIMA             21     0    2  1.000000
AVG(B,G,L)        12     3    8  0.800000
AVG(G,L)           9     3   11  0.750000
STK_LSTM(B,G,L)    8     3   12  0.727273
G                  6     3   14  0.666667
STK_ANN(B,G,L)     6     3   14  0.666667
STK_GRU(G,L)       6     3   14  0.666667
STK_GRU(B,L)       5     3   15  0.625000
AVG(B,L)           4     4   15  0.500000
STK_LSTM(B,G)      3     3   17  0.500000
STK_LSTM(B,L)      3     3   17  0.500000
AVG(B,G)           3     4   16  0.428571
STK_GRU(B,G,L)     3     6   14  0.333333
STK_GRU(B,G)       3     6   14  0.333333
L                  2     5   16  0.285714
STK_ANN(G,L)       2    10   11  0.166667
GRU-256            1     4   18  0.200000
STK_ANN(B,G)       1    12   10  0.076923
XGBoost            0    10   13  0.000000
STK_LSTM(G,L)      0    14    9  0.000000
B                  0    19    4  0

In [ ]:
val, test, wl, order = run_ticker(1120,calendar_ffill=True)
print(order)                                   # (0, 1, 2)
print(test.round(4).sort_values('RMSE'))
print(wl)   # ARIMA 1.2791, RW drift 1.2791, Naive 1.2802, G 1.2958, ...

E0000 00:00:1793940633.151262 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


BiRNN 128        epochs  118  val 0.001302


E0000 00:00:1793940666.937413 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


GRU 16           epochs  319  val 0.001216


E0000 00:00:1793940770.411957 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


LSTM 16          epochs  469  val 0.001233


... (54 outputs left out)

E0000 00:00:1793941691.869723 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
E0000 00:00:1793941700.472780 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),

GRU 256          epochs  115  val 0.001247


E0000 00:00:1793941736.876373 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


(0, 1, 2)
               MAE     MSE    RMSE     MRE
RW drift    0.6862  1.0807  1.0396  0.0123
Naive       0.6855  1.0820  1.0402  0.0123
ARIMA       0.6910  1.0827  1.0405  0.0124
G           0.7340  1.1507  1.0727  0.0133
AVG(B,G,L)  0.7338  1.1530  1.0738  0.0132


In [ ]:
val, test, wl, order = run_ticker(1120) #

E0000 00:00:1793941926.148451 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


BiRNN 128        epochs   37  val 0.001668


E0000 00:00:1793941933.098032 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


GRU 16           epochs  349  val 0.001482


E0000 00:00:1793941977.928212 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


LSTM 16          epochs  656  val 0.001545


... (54 outputs left out)

E0000 00:00:1793943005.289629 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
E0000 00:00:1793943013.173787 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),

GRU 256          epochs  127  val 0.001600


E0000 00:00:1793943029.249673 1470447 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


(0, 1, 2)
                     MAE       MSE     RMSE     MRE
RW drift          0.9026    1.6361   1.2791  0.0162
ARIMA             0.9025    1.6362   1.2791  0.0162
Naive             0.9027    1.6389   1.2802  0.0162
G                 0.9251    1.6790   1.2958  0.0166
AVG(G,L)          0.9323    1.7092   1.3074  0.0168
AVG(B,G,L)        0.9549    1.7670   1.3293  0.0171
AVG(B,G)          0.9622    1.7836   1.3355  0.0173
L                 0.9725    1.8461   1.3587  0.0174
AVG(B,L)          0.9982    1.9005   1.3786  0.0179
GRU-256           1.0253    2.0649   1.4370  0.0183
B                 1.0751    2.1452   1.4647  0.0192
XGBoost           1.1005    2.4221   1.5563  0.0196
STK_ANN(B,G,L)    1.2497    3.0581   1.7487  0.0216
STK_ANN(B,L)      1.2685    3.1097   1.7634  0.0220
RW (paper)        1.2348    3.1211   1.7667  0.0223
STK_GRU(B,G)      1.2569    3.1367   1.7711  0.0215
STK_GRU(B,L)      1.2754    3.1695   1.7803  0.0220
STK_ANN(B,G)      1.2807    3.1924   1.7867  0.0219
ST

In [41]:
print(order)                                   # (0, 1, 2)
print(test.round(4).sort_values('RMSE'))
print(wl) 

(0, 1, 2)
                     MAE       MSE     RMSE     MRE
RW drift          0.9026    1.6361   1.2791  0.0162
ARIMA             0.9025    1.6362   1.2791  0.0162
Naive             0.9027    1.6389   1.2802  0.0162
G                 0.9251    1.6790   1.2958  0.0166
AVG(G,L)          0.9323    1.7092   1.3074  0.0168
AVG(B,G,L)        0.9549    1.7670   1.3293  0.0171
AVG(B,G)          0.9622    1.7836   1.3355  0.0173
L                 0.9725    1.8461   1.3587  0.0174
AVG(B,L)          0.9982    1.9005   1.3786  0.0179
GRU-256           1.0253    2.0649   1.4370  0.0183
B                 1.0751    2.1452   1.4647  0.0192
XGBoost           1.1005    2.4221   1.5563  0.0196
STK_ANN(B,G,L)    1.2497    3.0581   1.7487  0.0216
STK_ANN(B,L)      1.2685    3.1097   1.7634  0.0220
RW (paper)        1.2348    3.1211   1.7667  0.0223
STK_GRU(B,G)      1.2569    3.1367   1.7711  0.0215
STK_GRU(B,L)      1.2754    3.1695   1.7803  0.0220
STK_ANN(B,G)      1.2807    3.1924   1.7867  0.0219
ST

In [ ]:
for name, sym in SYMBOLS.items():
    if os.path.exists(f'results/{name}_winloss.csv'):
        continue                                  # already done: allows resuming after a crash
    print('=' * 20, name)
    val, test, wl, order = run_ticker(sym,calendar_ffill=True)
    val.to_csv(f'results/{name}_val.csv')
    test.to_csv(f'results/{name}_test.csv')
    wl.to_csv(f'results/{name}_winloss.csv')
    print(name, 'ARIMA', order)